In [ ]:
# import sys
# print(sys.executable)

c:\T01\Agent01\.venv\Scripts\python.exe


In [53]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from IPython.display import Markdown, display, HTML
import json
from langchain_community.utilities import GoogleSerperAPIWrapper
import ipywidgets as widgets
import re
from playwright.sync_api import sync_playwright
import uuid
import requests
import json

In [ ]:
load_dotenv(override=True)

True

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
os.getenv('GOOGLE_API_KEY')

FINTO_BASE = "https://finto.day"
finto_email = os.environ.get("FINTO_EMAIL")
finto_password = os.environ.get("FINTO_PASSWORD")


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

In [58]:
serper = GoogleSerperAPIWrapper()
serper_images = GoogleSerperAPIWrapper(type="images")

In [61]:
prompt = """
You are a content agent responsible for producing and publishing content 
for our platform. Given a category and subtopic, follow this process 
IN ORDER, using the tools available to you:

Category must be exactly one of these (use the exact spelling/casing):
Technology, Web Development, Artificial Intelligence, Gadgets,
Business, Startups, Finance,
Lifestyle, Health, Travel

1. RESEARCH
   Call web_search tool with a short, specific query (4-6 words) to gather 
   current, factual context on the subtopic within the category. This is 
   to avoid generic or outdated filler — do not skip this step.
   You may call it up to 2 times if the first results are too broad or irrelevant.

2. WRITE CONTENT
   Using the research, write a piece with:
   - title (SEO-friendly, max 70 characters)
   - intro (2-3 sentences)
   - sections (5, each with a heading and body text)
   - conclusion (2-3 sentences)
   - tags (3-5 relevant SEO tags)
   Do not fabricate facts, statistics, or quotes not supported by the research.
   Word count target: word_count words total.

3. SOURCE IMAGES
   Call image_search tool once per needed image (3-5 total):
   - 1 hero/featured image — landscape orientation
   - 2-4 supporting images, one per relevant section - adjust accordingly that it do not end up taking more space the text content
   Only use royalty-free sources. Return the image URL and source name for each.
   If no suitable image is found for a section, skip it rather than 
   inventing a URL.
   - return the URL of each image used

4. ASSEMBLE DRAFT
   Combine the written content and images into a single JSON object 
   matching this structure:
   {
     "title": "", "slug": "", "category": "", "tags": [],
     "meta_description": "", "intro": "",
     "sections": [{"heading": "", "text": "", "image": {"url": "", "source": ""}}],
     "conclusion": "", "featured_image": {"url": "", "source": ""},
     "status": "draft"
   }

5. STOP FOR HUMAN APPROVAL
   Do NOT call publish tool yet. Present the assembled draft as your final 
   response for this turn, clearly labeled, and wait for explicit approval 
   before publishing.

6. PUBLISH (only after approval is given )
   Call publish tool with the approved payload. Set "status" to "draft" or 
   "live" based on what the human specifies.

7. REPORT
   After publish tool returns, report back the URL/ID and a 1-line summary.

Rules:
- Follow the steps in order — do not skip research or jump straight to writing.
- Never call publish tool without explicit human approval in the conversation.
- If any tool call fails, report the error and stop — do not retry more than once.
- Do not fabricate image URLs, facts, or statistics.
"""

In [62]:
def web_search(query: str)->str:
    """ Search the web for the current information on a given query"""
    return serper.run(query)

In [63]:
web_search_json = {
    "name": "web_search",
    "description": "Search the web for current information relevant to the topic",
    "parameters":{ 
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to seach the web(4-6 words)"
            } 
        },
    "required": ["query"],
    "additionalProperties": False
    }
}

In [64]:
def image_search(query):
    """Search for royalty-free images relevant to the query."""
    results = serper_images.results(query) 

    images = results.get("images", [])[:5]  
    if not images:
        return []

    return [
        {"url": img.get("imageUrl"), "source": img.get("source", "Unknown")}
        for img in images
    ]

In [65]:
image_search_json={
    "name": "image_search",
    "description": "Search the web for royalty free images on the platforms like unsplash, pixabay,p exels, etc relevant to the query",
    "parameters":{
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to search for images(4-6 words)"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [66]:
CATEGORY_MAP = {
    "Technology": 1, "Web Development": 2, "Artificial Intelligence": 3, "Gadgets": 4,
    "Business": 5, "Startups": 6, "Finance": 7,
    "Lifestyle": 8, "Health": 9, "Travel": 10,
}

def get_csrf_token(session, url):
    resp = session.get(url)
    match = re.search(r'name="_token" value="([^"]+)"', resp.text)
    if not match:
        raise ValueError(f"Could not find CSRF token on {url}")
    return match.group(1)

In [67]:
def login(session):
    login_url = f"{FINTO_BASE}/writer/login"
    token = get_csrf_token(session, login_url)
    resp = session.post(login_url, data={
        "_token": token,
        "email": finto_email,
        "password": finto_password,
    })
    resp.raise_for_status()
    if "/writer/login" in resp.url:
        raise ValueError("Login failed — check credentials or CSRF handling")

In [68]:
def download_image(url):
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.content

In [69]:
def format_body_for_finto(payload):
    parts = [payload.get("intro", "")]
    for section in payload.get("sections", []):
        parts.append(f"<h2>{section['heading']}</h2><p>{section['text']}</p>")
    parts.append(f"<p>{payload.get('conclusion', '')}</p>")
    return "".join(parts)

In [106]:
def build_acknowledgement(payload):
    sources = set()
    if payload.get("featured_image", {}).get("source"):
        sources.add(payload["featured_image"]["source"])
    for s in payload.get("sections", []):
        if s.get("image", {}).get("source"):
            sources.add(s["image"]["source"])
    return f"Images sourced from {', '.join(sources)}" if sources else ""

In [107]:
def publish(payload):
    """Publish the content draft to finto.day."""
    try:
        session = requests.Session()
        login(session)

        new_article_url = f"{FINTO_BASE}/writer/articles/create"
        token = get_csrf_token(session, new_article_url)

        body_html = format_body_for_finto(payload)

        data = {
            "_token": token,
            "title": payload["title"][:150],
            "short_description": payload.get("meta_description", "")[:255],
            "category_id": CATEGORY_MAP.get(payload.get("category"), ""),
            "body": body_html,
            "meta_keywords": ", ".join(payload.get("tags", [])),
            "meta_description": payload.get("meta_description", "")[:255],
            "meta_content": payload.get("intro", "")[:500],
            "acknowledgement": build_acknowledgement(payload),
            "is_published": "1" if payload.get("status") == "live" else "0",
            "is_full_width_image": "0",
            "image_gallery_layout": "vertical",
            "default_image": "new:0",
        }
        for i, tag in enumerate(payload.get("tags", [])[:5]):
            data[f"tags[{i}]"] = tag

        files = {}
        images = []
        if payload.get("featured_image", {}).get("url"):
            images.append(payload["featured_image"])
        for section in payload.get("sections", []):
            if section.get("image", {}).get("url"):
                images.append(section["image"])

        for i, img in enumerate(images[:5]):
            img_bytes = download_image(img["url"])
            files[f"images[{i}]"] = (f"image{i}.jpg", img_bytes, "image/jpeg")
            data[f"images_alt[{i}]"] = img.get("source", "")

        resp = session.post(f"{FINTO_BASE}/writer/articles", data=data, files=files)
        resp.raise_for_status()

        return {"success": True, "url": resp.url}

    except Exception as e:
        return {"success": False, "error": str(e)}

In [108]:
publish_json={
    "name": "publish",
    "description": "Publish the finished content draft to the platform.",
    "parameters":{
        "type": "object",
        "properties":{
            "payload":{
                "type": "object",
                "description": "The final content draft + images to publish"
            }
        },
        "required": ["payload"],
        "additionalProperties": False
    }
}

In [109]:
tools = [{"type": "function", "function": web_search_json}, {"type": "function", "function": image_search_json}, {"type": "function", "function": publish_json}]

In [110]:
tools_flow = {
    "web_search": web_search,
    "image_search": image_search,
    "publish": publish,
}                         

In [111]:
def agent01(category, subtopic, word_count):
    filled_prompt = prompt.replace("word_count", str(word_count))  # or however you're filling it
    messages = [
        {"role": "system", "content": filled_prompt},
        {"role": "user", "content": f"category: {category}, subtopic: {subtopic}"}
    ]
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[function_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    return response.choices[0].message.content, messages

In [112]:
def review_draft(messages, decision, feedback=None, live=False):
    if decision == "reject":
        if not feedback:
            raise ValueError("Feedback is required when rejecting a draft.")
        user_msg = f"Not approved. Please revise the draft based on this feedback: {feedback}"
    elif decision == "approve":
        user_msg = f"Approved. Publish with status = {'live' if live else 'draft'}."
    else:
        raise ValueError("decision must be 'approve' or 'reject'")

    messages.append({"role": "user", "content": user_msg})
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[fn_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    return response.choices[0].message.content, messages

In [113]:
def clean_json_string(s):
    s = s.strip()

    # strip markdown code fences if present
    if s.startswith("```"):
        s = s.split("\n", 1)[1]
        s = s.rsplit("```", 1)[0]
        s = s.strip()

    # extract only the first complete {...} block, ignore any trailing text
    match = re.search(r'\{.*\}', s, re.DOTALL)
    if match:
        s = match.group(0)

    return s.strip()

In [114]:

def display_draft(draft_json_str):
    draft_json_str = clean_json_string(draft_json_str)
    data = json.loads(draft_json_str)
 
    title = data.get('title') or '(no title)'
    meta_description = data.get('meta_description') or ''
    category = data.get('category') or ''
    tags = data.get('tags') or []
    intro = data.get('intro') or ''
    sections = data.get('sections') or []
    conclusion = data.get('conclusion') or ''
    status = data.get('status') or 'unknown'
    featured = data.get('featured_image') or {}
 
    # collect all images: featured first, then any section images, dedup + skip empty
    all_images = []
    if featured.get('url'):
        all_images.append(featured['url'])
    for section in sections:
        img = section.get('image') or {}
        url = img.get('url')
        if url and url not in all_images:
            all_images.append(url)
 
    gallery_id = f"gallery_{uuid.uuid4().hex[:8]}"
 
    # Build gallery HTML (hero + vertical thumbnail strip)
    if all_images:
        thumbs_html = "".join(
            f'<img src="{url}" class="{gallery_id}-thumb" '
            f'onclick="document.getElementById(\'{gallery_id}-hero\').src=\'{url}\'; '
            f'document.querySelectorAll(\'.{gallery_id}-thumb\').forEach(t=>t.classList.remove(\'active\')); '
            f'this.classList.add(\'active\')" />'
            for url in all_images
        )
        gallery_html = f"""
        <div style="display:flex; gap:12px; margin:16px 0;">
            <div style="flex:1; max-width:600px;">
                <img id="{gallery_id}-hero" src="{all_images[0]}"
                     style="width:100%; border-radius:8px; object-fit:cover; aspect-ratio:16/9;" />
            </div>
            <div style="display:flex; flex-direction:column; gap:8px; width:90px;">
                {thumbs_html}
            </div>
        </div>
        <style>
            .{gallery_id}-thumb {{
                width:100%; height:60px; object-fit:cover; border-radius:6px;
                cursor:pointer; opacity:0.65; border:2px solid transparent;
                transition: all 0.15s ease;
            }}
            .{gallery_id}-thumb:hover {{ opacity:1; }}
            .{gallery_id}-thumb.active {{ opacity:1; border-color:#4a90d9; }}
        </style>
        """
    else:
        gallery_html = "<p><em>(no images sourced for this draft)</em></p>"
 
    # Build sections HTML (text only now, no per-section image) 
    sections_html = ""
    for section in sections:
        heading = section.get('heading') or ''
        text = section.get('text') or ''
        sections_html += f"<h3>{heading}</h3><p>{text}</p>"
 
    html = f"""
    <div style="font-family:sans-serif; max-width:700px; line-height:1.5;">
        <h1>{title}</h1>
        <p><em>{meta_description}</em></p>
        <p><strong>Category:</strong> {category} | <strong>Tags:</strong> {', '.join(tags)}</p>
 
        {gallery_html}
 
        <p>{intro}</p>
        {sections_html}
        <p>{conclusion}</p>
        <hr/>
        <p><strong>Status:</strong> {status}</p>
    </div>
    """
 
    display(HTML(html))


In [94]:
# test_payload = {
#     "title": "Test Post - Please Ignore",
#     "category": "Technology",
#     "meta_description": "Testing the publish pipeline.",
#     "tags": ["test"],
#     "intro": "This is a test.",
#     "sections": [],
#     "conclusion": "End of test.",
#     "featured_image": {"url": "", "source": ""},
#     "status": "draft"
# }

# result = publish(test_payload)
# print(result)

In [117]:
draft, messages = agent01(category="Health", subtopic="food and diet in 2026", word_count=800)
display_draft(draft)

# print(repr(draft))

In [ ]:
data = json.loads(clean_json_string(draft))
print(data.get("featured_image"))
for s in data.get("sections", []):
    print(s["heading"], "->", s["image"])

{'url': 'https://qubit.capital/wp-content/uploads/2025/02/ChatGPT-Image-Nov-17-2025-12_45_03-PM_11zon.webp', 'source': 'Qubit Capital'}
Establish a Realistic Budget -> {'url': 'https://www.gofrugal.com/sites/blog/files/gofrugal/sample-5_1.jpg', 'source': 'Gofrugal'}
Prioritize Emergency Funds -> {'url': 'https://redfworkshop.org/wp-content/uploads/2023/10/Growth-Plan-Elements-Graphic.png', 'source': 'REDF Workshop'}
Master Debt and Credit Management -> None
Invest in Future Growth -> {'url': 'https://www.kitces.com/wp-content/uploads/2014/02/Five-Stages-Of-Growth-Graphic.png', 'source': 'Kitces.com'}
Maintain Tax-Aware Financial Planning -> None


In [118]:
revised_draft, messages = review_draft(messages, decision="reject", feedback="1st image is not visible? not loading!! rest of the four images are fine.")
# print(repr(revised_draft))
display_draft(revised_draft) 



In [119]:
result, messages = review_draft(messages, decision="approve", live=True)
print(result)

The content has been successfully published to the platform.

**Summary:** Your article "Food and Diet Trends Shaping 2026: A Guide to Healthy Living" is now live at https://finto.day/writer/articles/33/edit. It covers key 2026 trends including fiber-rich nutrition, functional ingredients, sustainability, and the rise of high-quality, convenient meal solutions.
